In [1]:
#Imports
from PIL import Image
import pandas as pd
from pathlib import Path
import re
import os
import numpy as np
from matplotlib import pyplot as plt

In [2]:
def scan_dir(directory, file_paths):
    for entry in os.scandir(directory):
        if entry.is_dir():
            scan_dir(entry.path, file_paths)
        elif entry.is_file() and ".ods" not in entry.name and ".yaml" not in entry.name:
            file_paths.append(entry.path)
   
    return file_paths

In [3]:
directory = "C:\\Users\\Installer\\Documents\\Data Science Class\\Computer-Vision-Pipeline\\data\\license_plate_detection"

file_paths = []
file_paths = scan_dir(directory, file_paths)

txt_paths = []
jpg_paths = []
names = []
for file in file_paths:
    if '.txt' in file:
        txt_paths.append(file)
    elif '.jpg' in file:
        jpg_paths.append(file)
        splits = re.split(r'\\|\.', file)
        with Image.open(file) as img:
                width, height = img.size
                name = {
                     "name": splits[-2],
                     "width": int(width),
                     "height": int(height)
                }
        names.append(name)

In [4]:
#Creates the dataframe for the bounding boxes in each image
crops = []
name_lookup = {n["name"]: n for n in names}

for file in txt_paths:
    stem = Path(file).stem

    if stem not in name_lookup:
        continue

    name = name_lookup[stem]
    with Path.open(file) as f:
        next(f) #Skips first line in file
        for line in f:
            line_split = line.split()
            crop = {
                "name": name["name"],
                "class": int(line_split[0]),
                "X Center": float(line_split[1]) * name["width"],
                "Y Center": float(line_split[2]) * name["height"],
                "Width": float(line_split[3]) * name["width"],
                "Height": float(line_split[4]) * name["height"]
            }
            crops.append(crop)

df = pd.DataFrame(crops)
df["x min"] = df["X Center"] - df["Width"]/2
df["y min"] = df["Y Center"] - df["Height"]/2
df["x max"] = df["X Center"] + df["Width"]/2
df["y max"] = df["Y Center"] + df["Height"]/2
df = df.drop(columns=["X Center", "Y Center", "Width", "Height"])

df

,name,class,x min,y min,x max,y max
0,lp_test_001,0,317.0,498.0,413.00,526.160
1,lp_test_001,0,875.0,385.0,924.28,406.120
2,lp_test_002,0,406.0,712.0,492.40,748.484
3,lp_test_003,0,435.0,528.0,477.24,549.120
4,lp_test_004,0,42.0,380.0,72.72,417.784
...,...,...,...,...,...,...
10532,lp_valid_995,0,195.0,229.0,247.00,241.000
10533,lp_valid_996,0,218.0,203.0,280.00,212.000
10534,lp_valid_997,0,198.0,137.0,253.00,151.000
10535,lp_valid_998,0,225.0,172.0,267.00,182.000


In [ ]:
# Function to create image crops locally
for file in jpg_paths:
    stem = Path(file).stem
    

